# SE3082 Parallel Computing — Lab 10
Student: IT24103467. Run on a T4 GPU. Outputs are from actual Colab runs, not sample data.

In [ ]:
!nvidia-smi
!nvcc --version

In [ ]:
%%writefile cuda_utils.cuh
#ifndef CUDA_UTILS_CUH
#define CUDA_UTILS_CUH
#include <cuda_runtime.h>
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define CHECK(call) do { \
    cudaError_t e = (call); \
    if (e != cudaSuccess) { \
        fprintf(stderr, "Error: %s:%d, code:%d, reason: %s\n", \
                __FILE__, __LINE__, (int)e, cudaGetErrorString(e)); \
        exit(EXIT_FAILURE); \
    } \
} while (0)

inline double cpu_second() {
    timeval t;
    gettimeofday(&t, nullptr);
    return t.tv_sec + t.tv_usec * 1e-6;
}

inline void *host_alloc(size_t n) {
    void *p = malloc(n);
    if (!p) { fprintf(stderr, "Host allocation failed: %zu bytes\n", n); exit(1); }
    return p;
}

inline void random_ints(int *p, size_t n) {
    for (size_t i = 0; i < n; ++i) p[i] = rand() % 100;
}
#endif


## Exercise 1: GPU timing
Three trials for each Lab 9 program, plus 100-million-element transfer comparison. The clock stops only after `cudaDeviceSynchronize()`.

In [ ]:
%%writefile ex1_timing.cu
#include "cuda_utils.cuh"

__global__ void add_blocks(const int *a, const int *b, int *c) {
    int i = blockIdx.x;
    c[i] = a[i] + b[i];
}
__global__ void add_threads(const int *a, const int *b, int *c) {
    int i = threadIdx.x;
    c[i] = a[i] + b[i];
}
__global__ void multiply_vector(const int *a, const int *b, int *c, size_t n) {
    size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) c[i] = a[i] * b[i];
}
__global__ void multiply_matrix(const int *a, const int *b, int *c,
                                int rows, int cols) {
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;
    if (x < cols && y < rows) {
        size_t i = (size_t)y * cols + x;
        c[i] = a[i] * b[i];
    }
}

void run_case(const char *label, size_t n, int mode) {
    size_t bytes = n * sizeof(int);
    int *a = (int *)host_alloc(bytes), *b = (int *)host_alloc(bytes);
    int *c = (int *)host_alloc(bytes);
    random_ints(a, n); random_ints(b, n);
    int *da, *db, *dc;
    CHECK(cudaMalloc((void **)&da, bytes));
    CHECK(cudaMalloc((void **)&db, bytes));
    CHECK(cudaMalloc((void **)&dc, bytes));
    printf("%s: %zu elements\n", label, n);
    for (int rep = 1; rep <= 3; ++rep) {
        double whole_start = cpu_second();
        CHECK(cudaMemcpy(da, a, bytes, cudaMemcpyHostToDevice));
        CHECK(cudaMemcpy(db, b, bytes, cudaMemcpyHostToDevice));
        double kernel_start = cpu_second();
        if (mode == 0) add_blocks<<<512, 1>>>(da, db, dc);
        else if (mode == 1) add_threads<<<1, 512>>>(da, db, dc);
        else if (mode == 2) multiply_vector<<<(n + 511) / 512, 512>>>(da, db, dc, n);
        else multiply_matrix<<<dim3(313, 625), dim3(32, 16)>>>(da, db, dc, 10000, 10000);
        CHECK(cudaGetLastError());
        CHECK(cudaDeviceSynchronize());
        double kernel_time = cpu_second() - kernel_start;
        CHECK(cudaMemcpy(c, dc, bytes, cudaMemcpyDeviceToHost));
        double whole_time = cpu_second() - whole_start;
        size_t errors = 0;
        for (size_t i = 0; i < n; ++i)
            errors += c[i] != (mode <= 1 ? a[i] + b[i] : a[i] * b[i]);
        printf("run %d: kernel=%.7f s, copy+kernel+copy=%.7f s, mismatches=%zu\n",
               rep, kernel_time, whole_time, errors);
        if (errors) exit(1);
    }
    CHECK(cudaFree(da)); CHECK(cudaFree(db)); CHECK(cudaFree(dc));
    free(a); free(b); free(c);
}

int main() {
    run_case("Lab 9 Ex 3: 512 blocks x 1 thread", 512, 0);
    run_case("Lab 9 Ex 4: 1 block x 512 threads", 512, 1);
    run_case("Lab 9 Ex 5: 10 million vector", 10000000, 2);
    run_case("Lab 9 Ex 6: 100 million matrix", 100000000, 3);
    run_case("Transfer comparison: 100 million vector", 100000000, 2);
    return 0;
}


In [ ]:
!nvcc -arch=sm_75 ex1_timing.cu -o ex1_timing && ./ex1_timing

The one-block version should avoid the overhead of scheduling 512 tiny blocks, but both short timings may be noisy. The 100-million-element case does more work, though timing need not scale exactly tenfold due to memory throughput and launch overhead. Whole-sequence time includes host/device transfers and should be substantially larger than kernel-only time. Without synchronizing before stopping the clock, the apparent kernel time measures mostly launch overhead; the later blocking copy can still make verification pass.

## Exercise 2: CHECK error handling
The intentional failures are expected and are run under shell guards so the notebook continues. `invalid` probes the launch configuration; `oom` probes device allocation.

In [ ]:
%%writefile ex2_check.cu
#include "cuda_utils.cuh"
#include <cstring>

__global__ void add(const int *a, const int *b, int *c) {
    c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}

int main(int argc, char **argv) {
    const char *test = argc > 1 ? argv[1] : "normal";
    const int n = 512;
    const size_t bytes = n * sizeof(int);
    int *da, *db, *dc;
    // These modes intentionally fail to prove CHECK reports errors.
    if (!strcmp(test, "oom")) CHECK(cudaMalloc((void **)&da, (size_t)1 << 50));
    CHECK(cudaMalloc((void **)&da, bytes));
    CHECK(cudaMalloc((void **)&db, bytes));
    CHECK(cudaMalloc((void **)&dc, bytes));
    int a[n], b[n], c[n];
    random_ints(a, n); random_ints(b, n);
    CHECK(cudaMemcpy(da, a, bytes, cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(db, b, bytes, cudaMemcpyHostToDevice));
    double start = cpu_second();
    if (!strcmp(test, "invalid")) add<<<n, 2048>>>(da, db, dc);
    else add<<<n, 1>>>(da, db, dc);
    CHECK(cudaGetLastError());
    CHECK(cudaDeviceSynchronize());
    double elapsed = cpu_second() - start;
    CHECK(cudaMemcpy(c, dc, bytes, cudaMemcpyDeviceToHost));
    int errors = 0;
    for (int i = 0; i < n; ++i) errors += c[i] != a[i] + b[i];
    printf("Verification: %d mismatches out of %d\nTime taken - %.7f\n", errors, n, elapsed);
    CHECK(cudaFree(da)); CHECK(cudaFree(db)); CHECK(cudaFree(dc));
    return errors ? 1 : 0;
}


In [ ]:
!nvcc -arch=sm_75 ex2_check.cu -o ex2_check && ./ex2_check
!./ex2_check invalid || echo 'Expected invalid-configuration failure'
!./ex2_check oom || echo 'Expected out-of-memory failure'

`CHECK(cudaGetLastError())` catches invalid launch configurations; `CHECK(cudaMalloc(...))` catches allocation errors; `CHECK(cudaDeviceSynchronize())` catches execution failures. Without `CHECK`, an invalid launch might be noticed only through incorrect verification results.

## Exercise 3: 800 × 600 image
A 50 × 38 grid of 16 × 16 threads covers every pixel. Values above 127 saturate to 255 when doubled, so roughly half the pixels are capped.

In [ ]:
%%writefile ex3_picture.cu
#include "cuda_utils.cuh"

constexpr int WIDTH = 800, HEIGHT = 600;

__global__ void picture_kernel(const unsigned char *in, unsigned char *out,
                               int width, int height) {
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    if (row < height && col < width) {
        int i = row * width + col;
        int doubled = 2 * in[i];
        out[i] = (unsigned char)(doubled > 255 ? 255 : doubled);
    }
}

int main() {
    size_t count = (size_t)WIDTH * HEIGHT;
    size_t bytes = count * sizeof(unsigned char);
    auto *in = (unsigned char *)host_alloc(bytes);
    auto *out = (unsigned char *)host_alloc(bytes);
    for (size_t i = 0; i < count; ++i) in[i] = rand() % 256;
    unsigned char *din, *dout;
    CHECK(cudaMalloc((void **)&din, bytes));
    CHECK(cudaMalloc((void **)&dout, bytes));
    CHECK(cudaMemcpy(din, in, bytes, cudaMemcpyHostToDevice));
    dim3 threads(16, 16);
    dim3 blocks((WIDTH + threads.x - 1) / threads.x,
                (HEIGHT + threads.y - 1) / threads.y);
    printf("Grid: %u x %u blocks of %u x %u threads\n",
           blocks.x, blocks.y, threads.x, threads.y);
    double start = cpu_second();
    picture_kernel<<<blocks, threads>>>(din, dout, WIDTH, HEIGHT);
    CHECK(cudaGetLastError());
    CHECK(cudaDeviceSynchronize());
    double elapsed = cpu_second() - start;
    CHECK(cudaMemcpy(out, dout, bytes, cudaMemcpyDeviceToHost));
    puts("Original top-left 50 x 50:");
    for (int y = 0; y < 50; ++y) {
        for (int x = 0; x < 50; ++x) printf("%3d ", in[y * WIDTH + x]);
        putchar('\n');
    }
    puts("Updated top-left 50 x 50:");
    for (int y = 0; y < 50; ++y) {
        for (int x = 0; x < 50; ++x) printf("%3d ", out[y * WIDTH + x]);
        putchar('\n');
    }
    size_t errors = 0, capped = 0;
    for (size_t i = 0; i < count; ++i) {
        int doubled = 2 * in[i];
        capped += doubled > 255;
        errors += out[i] != (doubled > 255 ? 255 : doubled);
    }
    printf("Verification: %zu mismatches out of %zu\nPixels capped: %zu\nTime taken - %.7f\n",
           errors, count, capped, elapsed);
    CHECK(cudaFree(din)); CHECK(cudaFree(dout));
    free(in); free(out);
    return errors ? 1 : 0;
}


In [ ]:
!nvcc -arch=sm_75 ex3_picture.cu -o ex3_picture && ./ex3_picture

## Exercise 4: 500 × 500 × 500 volume
An 8 × 8 × 8 block and 63 × 63 × 63 grid cover the volume with 3D bounds checks. This is element-wise multiplication, not linear-algebra matrix multiplication.

In [ ]:
%%writefile ex4_volume.cu
#include "cuda_utils.cuh"

constexpr int NX = 500, NY = 500, NZ = 500;

__global__ void multiply_volume(const int *a, const int *b, int *c,
                                int nx, int ny, int nz) {
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;
    int z = blockIdx.z * blockDim.z + threadIdx.z;
    if (x < nx && y < ny && z < nz) {
        size_t i = ((size_t)z * ny + y) * nx + x;
        c[i] = a[i] * b[i];
    }
}

int main() {
    size_t count = (size_t)NX * NY * NZ;
    size_t bytes = count * sizeof(int);
    int *a = (int *)host_alloc(bytes), *b = (int *)host_alloc(bytes);
    int *c = (int *)host_alloc(bytes);
    random_ints(a, count); random_ints(b, count);
    int *da, *db, *dc;
    CHECK(cudaMalloc((void **)&da, bytes));
    CHECK(cudaMalloc((void **)&db, bytes));
    CHECK(cudaMalloc((void **)&dc, bytes));
    CHECK(cudaMemcpy(da, a, bytes, cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(db, b, bytes, cudaMemcpyHostToDevice));
    dim3 threads(8, 8, 8);
    dim3 blocks((NX + 7) / 8, (NY + 7) / 8, (NZ + 7) / 8);
    printf("Grid: %u x %u x %u blocks\n", blocks.x, blocks.y, blocks.z);
    double start = cpu_second();
    multiply_volume<<<blocks, threads>>>(da, db, dc, NX, NY, NZ);
    CHECK(cudaGetLastError());
    CHECK(cudaDeviceSynchronize());
    double elapsed = cpu_second() - start;
    CHECK(cudaMemcpy(c, dc, bytes, cudaMemcpyDeviceToHost));
    for (size_t i = count - 1000; i < count; ++i) {
        size_t z = i / (NX * NY), y = (i / NX) % NY, x = i % NX;
        printf("C[%zu][%zu][%zu] = %d x %d = %d\n", z, y, x, a[i], b[i], c[i]);
    }
    size_t errors = 0;
    for (size_t i = 0; i < count; ++i) errors += c[i] != a[i] * b[i];
    printf("Verification: %zu mismatches out of %zu\nTime taken - %.7f\n",
           errors, count, elapsed);
    CHECK(cudaFree(da)); CHECK(cudaFree(db)); CHECK(cudaFree(dc));
    free(a); free(b); free(c);
    return errors ? 1 : 0;
}


In [ ]:
!nvcc -arch=sm_75 ex4_volume.cu -o ex4_volume && ./ex4_volume

The 3D case does 125 million products versus 100 million in Lab 9's 2D case. Its time may differ for memory-access, scheduling, and cache reasons, not just the element count.